<a href="https://colab.research.google.com/github/LolloS8/Paper-Replication-/blob/main/Hedging_Paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

"Does Speculation in Futures Markets Improve Commodity Hedging Decision?"

The paper compares two approaches to risk management in commodity futures markets: traditional hedging and selective hedging.
As the paper shows at the beginning, the out-of-sample selective strategy fails to outperform traditional coverage.  
- Adrian Fernandez-Perez
- Ana-Maria Fuertes
- Joelle Miffre

In [1]:
import numpy as np
import pandas as pd

# 1. Calculation of the Traditional Hedging (MinVar) and Selective Hedging (HistAve).
def calculation_hedge_ratios(spot_returns, futures_retunrs, winodw = 520, gamma = 5):
  beta_t_list = [] # Traditional Hedging (MinaVar)
  h_t_list = [] # Selective Hedging (HistAve)

  n_obs = len(spot_returns) # make sure we have the same length

  # Rolling Window
  for t in range(winodw, n_obs):
    spot_window = spot_returns[t-window : t]
    fut_window = futures_returns[t-window : t]

    # Futures' Variance and Covariance (Spot, Future)
    cov_matrix = np.cov(spot_window, fut_window)
    cov_sf = cov_matrix[0, 1] # Covariance between spot and future
    var_f = cov_matrix[1, 1] # Futures' Variance

    # Traditional Hedging
    beta_t = cov_sf / var_f

    # Selective Hedging: Historical Average (HistAve)
    expected_return_fut = np.mean(fut_window)

    # Formula: bet_t - (E_t(delta( f) / (gamma * var_f))
    speculative_component = expected_return_fut / (gamma * var_f)
    h_t = beta_t - speculative_component

    # Save the result
    beta_t_list.append(beta_t)
    h_t_list.append(h_t)

  return beta_t_list, h_t_list

# Simulation
if __name__ == "main":
  np.random.seed(42)
  # 1000 Weeks with random returns
  spot_data = np.random.normal(0.001, 0.02, 1000)
  future_data = np.random.normal(0.001, 0.025, 1000) + 0.8 * spot_data

  beta, selective_h = calculation_hedge_ratios(spot_data, future_data, window = 50)
  print(f'First 5 values MinVar (beta_t): {[round(b,4) for b in beta[:5]]}')
  print(f'First 5 values Selective Hedging (h_t): {[round(h, 4) for h in selective_h[:5]]}')



In the second part, we will compare if the selective hedging startegy can generate a better utility than traditional one (MinVar)

In [8]:
import numpy as np
import pandas as pd

def calculate_hedging_effectiveness(spot_returns, futures_returns, beta_t_list, h_t_list, window = 520, gamma = 5):
  # -- Calculate the Expected Utility Gain for the Min Var POrtfolio & HistAve

  oos_spot = spot_returns[window:]
  oos_futures = futures_returns[window:]

  beta_t = np.array(beta_t_list[:-1])
  h_t = np.array(h_t_list[:-1])

  # Returns at time t+1
  ret_spot_t1 = oos_spot[1:]
  ret_fut_t1 = oos_futures[1:]

  returns_unhedged = ret_spot_t1
  returns_minvar = ret_spot_t1 - (beta_t * ret_fut_t1)
  returns_histave = ret_spot_t1 - (h_t * ret_fut_t1)

  return_histave = ret_spot_t1 - (h_t * ret_fut_t1)

  # Utility Function for Mean Variance (EQ. 1 in the paper)
  def calc_annualized_utility(returns, gamma):
    mean_ret = np.mean(returns) * 52
    var_ret = np.var(returns) * 52
    return mean_ret -(0.5 * gamma * var_ret)

  u_spot = calc_annualized_utility(returns_unhedged, gamma)
  u_minvar = calc_annualized_utility(returns_minvar, gamma)
  u_histave = calc_annualized_utility(return_histave, gamma)

  # Expected Utility Gain
  gain_minvar = u_minvar - u_spot
  gain_histave = u_histave - u_spot

  return gain_minvar, gain_histave, u_spot, u_minvar, u_histave

# Simulation
if __name__ == "__main__":
    np.random.seed(42)

    # Generate 1000 weeks of random returns
    spot_data = np.random.normal(0.001, 0.02, 1000)

    # Assume futures are correlated to spot but with some noise and a different mean
    future_data = np.random.normal(0.0005, 0.025, 1000) + 0.8 * spot_data

    # Parameters
    window_size = 50
    gamma_val = 5

    # 1. Call your function (assuming it's defined above in the file)
    # beta_t_list, h_t_list = calculation_hedge_ratios(spot_data, future_data, window=window_size, gamma=gamma_val)

    # (Mocking your function to run this script in isolation)
    def mock_calculation(spot, fut, w, g):
        return np.random.uniform(0.8, 1.0, len(spot)-w), np.random.uniform(0.6, 1.2, len(spot)-w)

    beta_t_list, h_t_list = mock_calculation(spot_data, future_data, window_size, gamma_val)

    # 2. Calculate the Expected Utility Gain
    gain_mv, gain_ha, u_s, u_mv, u_ha = calculate_hedging_effectiveness(
        spot_data,
        future_data,
        beta_t_list,
        h_t_list,
        window=window_size,
        gamma=gamma_val
    )

    print("--- HEDGING EFFECTIVENESS RESULTS ---")
    print(f"Annualized Spot Utility (Unhedged): {u_s:.4f}")
    print(f"Annualized MinVar Utility:          {u_mv:.4f}")
    print(f"Annualized HistAve Utility:         {u_ha:.4f}\n")

    print(f"Expected Utility Gain (MinVar):  {gain_mv:.4f} ({gain_mv*100:.2f}%)")
    print(f"Expected Utility Gain (HistAve): {gain_ha:.4f} ({gain_ha*100:.2f}%)")

    if gain_mv > gain_ha:
        print("\nConclusion: In this simulation, the traditional method (MinVar) beat speculation (HistAve), in line with the paper's conclusions.")
    else:
        print("\nConclusion: In this simulation, selective hedging (HistAve) outperformed MinVar.")


--- HEDGING EFFECTIVENESS RESULTS ---
Annualized Spot Utility (Unhedged): 0.0352
Annualized MinVar Utility:          -0.1345
Annualized HistAve Utility:         -0.1389

Expected Utility Gain (MinVar):  -0.1697 (-16.97%)
Expected Utility Gain (HistAve): -0.1741 (-17.41%)

Conclusion: In this simulation, the traditional method (MinVar) beat speculation (HistAve), in line with the paper's conclusions.
